In [ ]:
import pandas as pd

# Importing dataset and naming it as df
df = pd.read_csv("/content/seer colon cancer capstone 14-7-2026.txt")


In [ ]:
# checking df size and number of rows and columns
df.info()


In [ ]:
# Print all column names in the DataFrame
print("Variables in the dataset:")
for col in df.columns:
    print(col)


In [ ]:
# Keep only the variables the analysis needs and drop everything else
KEEP = {
    "Year of diagnosis":                                            "dx_year",

    # OUTCOME 1 — late stage
    "Combined Summary Stage with Expanded Regional Codes (2004+)":  "summary_stage",

    # OUTCOME 2 — surgery
    "Reason no cancer-directed surgery":                            "reason_no_surgery",

    # predictors, all four RQs
    "Age recode with <1 year olds and 90+":                         "age_recode",
    "Age recode with single ages and 90+":                          "age_single",
    "Sex":                                                          "sex",
    "Race and origin recode (NHW, NHB, NHAIAN, NHAPI, Hispanic)":   "race_eth",
    "Primary Site - labeled":                                       "primary_site",
    "Median household income inflation adj to 2024":                "county_income",
    "Rural-Urban Continuum Code":                                   "rucc",
    "Marital status at diagnosis":                                  "marital",

    # surgery models only (RQ3, RQ4)
    "Derived Summary Grade 2018 (2018+)":                           "grade",
    "Histologic Type ICD-O-3":                                      "histology",
    "Tumor Size Summary (2016+)":                                   "tumor_size",
}

df = df[list(KEEP)].rename(columns=KEEP)

df.info()

In [ ]:
# Print all column names in the DataFrame after choosing the needed variables
print("Variables in the dataset:")
for col in df.columns:
    print(col)

In [ ]:
df['dx_year'].value_counts(dropna=False)


In [ ]:
df['summary_stage'].value_counts(dropna=False)


In [ ]:
# Stage: rename levels to localized, regional, distant and unknown
stage_map = {
    "In situ":                                                      "Localized",
    "Localized only":                                               "Localized",
    "Regional by direct extension only":                            "Regional",
    "Regional lymph nodes involved only":                           "Regional",
    "Regional by both direct extension and lymph node involvement": "Regional",
    "Distant site(s)/node(s) involved":                             "Distant",
    "Unknown/unstaged/unspecified/DCO":                               np.nan,
}

df["stage"] = df["summary_stage"].map(stage_map)

# OUTCOME for RQ1/RQ2 which is late stage pressentation
df["late_stage"] = df["stage"].map({"Localized": 0, "Regional": 1, "Distant": 1})

print(df["stage"].value_counts(dropna=False))
print(df["late_stage"].value_counts(dropna=False, normalize=True))

In [ ]:
df['reason_no_surgery'].value_counts(dropna=False)


In [ ]:
# Surgery: did the patient receive cancer-directed surgery? Binary outcome.
surgery_map = {
    "Surgery performed":                                                            1,
    "Not recommended":                                                              0,
    "Not recommended, contraindicated due to other cond; autopsy only (1973-2002)": 0,
    "Recommended but not performed, patient refused":                               0,
    "Recommended but not performed, unknown reason":                                0,
    "Not performed, patient died prior to recommended surgery":                     0,
    "Recommended, unknown if performed":                                            np.nan,
    "Unknown; death certificate; or autopsy only (2003+)":                          np.nan,
}

df["surgery"] = df["reason_no_surgery"].map(surgery_map)

print(df["surgery"].value_counts(dropna=False))

In [ ]:
df['age_recode'].value_counts(dropna=False)


In [ ]:
# Age become 4 bands
age_map = {
    "15-19 years": "<50",   "20-24 years": "<50",   "25-29 years": "<50",
    "30-34 years": "<50",   "35-39 years": "<50",   "40-44 years": "<50",
    "45-49 years": "<50",
    "50-54 years": "50-64", "55-59 years": "50-64", "60-64 years": "50-64",
    "65-69 years": "65-74", "70-74 years": "65-74",
    "75-79 years": "75+",   "80-84 years": "75+",   "85-89 years": "75+",
    "90+ years":   "75+",
}

df["age_band"] = df["age_recode"].map(age_map)

df["age_band"] = pd.Categorical(
    df["age_band"],
    categories=["50-64", "<50", "65-74", "75+"],
    ordered=False,
)

print(df["age_band"].value_counts())

In [ ]:
# cleaning numerical age
df["age_num"] = (df["age_single"]
                 .str.extract(r"(\d+)")[0]
                 .astype(int))

In [ ]:
df['age_num'].value_counts(dropna=False)


In [ ]:
df['sex'].value_counts(dropna=False)


In [ ]:
df['race_eth'].value_counts(dropna=False)


In [ ]:
df['primary_site'].value_counts(dropna=False)


In [ ]:
# Cleaning site into proximal (right) vs distal (left) colon
site_map = {
    "C18.0-Cecum":                    "Proximal",
    "C18.2-Ascending colon":          "Proximal",
    "C18.3-Hepatic flexure of colon": "Proximal",
    "C18.4-Transverse colon":         "Proximal",
    "C18.5-Splenic flexure of colon": "Distal",
    "C18.6-Descending colon":         "Distal",
    "C18.7-Sigmoid colon":            "Distal",
}

df["subsite"] = df["primary_site"].map(site_map)

print(df["subsite"].value_counts())

In [ ]:
df['county_income'].value_counts(dropna=False)


In [ ]:
# County median household income, cleaned into 5 ordered bands
income_map = {
    "< $40,000":                              "< $60k",
    "$40,000 - $44,999":                      "< $60k",
    "$45,000 - $49,999":                      "< $60k",
    "$50,000 - $54,999":                      "< $60k",
    "$55,000 - $59,999":                      "< $60k",
    "$60,000 - $64,999":                      "$60-75k",
    "$65,000 - $69,999":                      "$60-75k",
    "$70,000 - $74,999":                      "$60-75k",
    "$75,000 - $79,999":                      "$75-90k",
    "$80,000 - $84,999":                      "$75-90k",
    "$85,000 - $89,999":                      "$75-90k",
    "$90,000 - $94,999":                      "$90-110k",
    "$95,000 - $99,999":                      "$90-110k",
    "$100,000 - $109,999":                    "$90-110k",
    "$110,000 - $119,999":                    "$110k+",
    "$120,000+":                              "$110k+",
    "Unknown/missing/no match/Not 1990-2024": np.nan,
}

df["income5"] = df["county_income"].map(income_map)

df["income5"] = pd.Categorical(
    df["income5"],
    categories=["< $60k", "$60-75k", "$75-90k", "$90-110k", "$110k+"],
    ordered=True,
)

print(df["income5"].value_counts(dropna=False).sort_index())

In [ ]:
df['rucc'].value_counts(dropna=False)


In [ ]:
# Rurality: metro vs non-metro
rucc_map = {
    "Counties in metropolitan areas ge 1 million pop":              "Metro",
    "Counties in metropolitan areas of 250,000 to 1 million pop":   "Metro",
    "Counties in metropolitan areas of lt 250 thousand pop":        "Metro",
    "Nonmetropolitan counties adjacent to a metropolitan area":     "NonMetro",
    "Nonmetropolitan counties not adjacent to a metropolitan area": "NonMetro",
    "Unknown/missing/no match (Alaska or Hawaii - Entire State)":   np.nan,
    "Unknown/missing/no match/Not 1990-2024":                       np.nan,
}

df["rurality"] = df["rucc"].map(rucc_map)

df["rurality"] = pd.Categorical(df["rurality"], categories=["Metro", "NonMetro"])

print(df["rurality"].value_counts(dropna=False))

In [ ]:
df['marital'].value_counts(dropna=False)


In [ ]:
# Marital status: married vs unmarried
marital_map = {
    "Married (including common law)": "Married",
    "Single (never married)":         "Unmarried",
    "Widowed":                        "Unmarried",
    "Divorced":                       "Unmarried",
    "Separated":                      "Unmarried",
    "Unmarried or Domestic Partner":  "Unmarried",
    "Unknown":                        np.nan,
}

df["marital"] = df["marital"].map(marital_map)

df["marital"] = pd.Categorical(df["marital"], categories=["Married", "Unmarried"])

print(df["marital"].value_counts(dropna=False))

In [ ]:
df['grade'].value_counts(dropna=False)


In [ ]:
# Grade: 2-levels low vs high
grade_map = {
    "Site-specific grade system category (1)": "Low",      # well differentiated
    "Site-specific grade system category (2)": "Low",      # moderately differentiated
    "Site-specific grade system category (3)": "High",     # poorly differentiated
    "Site-specific grade system category (4)": "High",     # undifferentiated
    "Grade cannot be assessed; Unknown":       np.nan,
}

df["grade"] = df["grade"].map(grade_map)

print(df["grade"].value_counts())

In [ ]:
df['histology'].value_counts(dropna=False)


In [ ]:
# Histology: 2 levels — conventional adenocarcinoma vs variant
hist_map = {
    8480: "Variant",   # mucinous
    8481: "Variant",   # mucinous
    8490: "Variant",   # signet-ring
}

# everything else in the cohort is conventional adenocarcinoma
df["histology"] = df["histology"].map(hist_map).fillna("AdenoNOS")

df["histology"] = pd.Categorical(
    df["histology"],
    categories=["AdenoNOS", "Variant"],
    ordered=False,
)

print(df["histology"].value_counts())

In [ ]:
df['tumor_size'].value_counts(dropna=False)


In [ ]:
import numpy as np

df["tumor_size"] = pd.to_numeric(df["tumor_size"], errors="coerce")

# SEER special codes that need to become np.nan
#   000 = no mass found            989      = >=989mm
#   990 = microscopic focus        991-998  = descriptive ("<2cm", "diffuse")
#   999 = unknown
df.loc[(df["tumor_size"] == 0) | (df["tumor_size"] >= 989), "tumor_size"] = np.nan

print(df["tumor_size"].describe())

In [ ]:
# Print all column names in the DataFrame
print("Variables in the dataset:")
for col in df.columns:
    print(col)


In [ ]:
# Drop the raw source columns because   the cleaned versions exist
df = df.drop(columns=[
    "summary_stage",       # -> stage, late_stage
    "reason_no_surgery",   # -> surgery
    "primary_site",        # -> subsite
    "age_recode",          # -> age_band
    "age_single",          # -> age_num
    "rucc",                # -> rurality
    "county_income",       # -> income5
    "dx_year",      # diagnosis year not needed
])


In [ ]:
df.info()

In [ ]:
df['sex'].value_counts(dropna=False)


In [ ]:
df['race_eth'].value_counts(dropna=False)


In [ ]:
df['marital'].value_counts(dropna=False)


In [ ]:
df['grade'].value_counts(dropna=False)


In [ ]:
df['histology'].value_counts(dropna=False)


In [ ]:
df['stage'].value_counts(dropna=False)


In [ ]:
df['late_stage'].value_counts(dropna=False)


In [ ]:
df['surgery'].value_counts(dropna=False)


In [ ]:
df['age_band'].value_counts(dropna=False)


In [ ]:
df['subsite'].value_counts(dropna=False)


In [ ]:
df['income5'].value_counts(dropna=False)


In [ ]:
df['rurality'].value_counts(dropna=False)


In [ ]:
df['tumor_size'].describe()


In [ ]:
df['age_num'].describe()
